# Video sampling logic

In [ ]:
import pickle
import torch
from pathlib import Path
import numpy as np
import random
import pandas as pd

In [ ]:
project_root = Path.cwd().parent
data_root = str(project_root / "multisports_data/data/trainval")
ground_truth_path= str(project_root / "multisports_fewshot_GT.pkl")

# Read The GT file
with open(ground_truth_path, "rb") as file:
    data = pickle.load(file)

# GTtubes
gttubes = data["gttubes"]

# Get the test classes
possible_classes = data["test_labels"]

# Splitting logic to query and support 
# Pick N novel classes
N = 5
# Pick at least M samples from those classes
M = 15
# NUmber of support samples
K = 5

# Sample classes
novel_classes_ids = random.sample(range(len(possible_classes)), N)

# Test videos
test_videos = data["test_videos"]

# For now do so that it is >= M, for results you can always only take M 

# Get the videos that include those classes and sample at least K 
video_dict = {} # video id and number of instances of the action in the class action : {video_id1: 4, video_id2: 4}


def get_m_vids_from_novel_class(samples_set, novel_classes_ids, M):

    for novel_class_id in novel_classes_ids:
        video_dict[novel_class_id] = {}

        # Go through all the videos in test videos
        for video in samples_set:
            tubes = gttubes[video]
            # Count the number of instances in the video
            if novel_class_id in gttubes[video].keys():
                instance_count = len(tubes[novel_class_id])
                video_dict[novel_class_id][video] = instance_count

    # Now sample those classes in order (they are already random so it's fine if we don't mix again)
    query_videos = set()
    accumulated_counts = {cls: 0 for cls in novel_classes_ids}

    for cls, dct in video_dict.items():

        # Shuffle the videos so that we pick random instances every time 
        video_items = list(dct.items())
        random.shuffle(video_items)

        for video, cnt in video_items:
            # Stop when enough instances 
            if accumulated_counts[cls] >= M:
                break
            # if not yet in query videos add it to the query videos list
            if video not in query_videos:
                query_videos.add(video)

                # Add the counts for all of the classes that are in the video
                for any_cls in novel_classes_ids:
                    if video in video_dict[any_cls]:
                        accumulated_counts[any_cls] += video_dict[any_cls][video]
                

    return list(query_videos), accumulated_counts


query_videos, accumulated_counts = get_m_vids_from_novel_class(test_videos, novel_classes_ids, M)


# Count the number of instances we go
for cls, cnts in accumulated_counts.items():
    print(f"{cnts} instances for class: {cls}")

# Pick K instances for the prototype - support set, from the remaining videos
remaining_vids = list(set(test_videos) - set(query_videos))
support_videos, accumulated_counts = get_m_vids_from_novel_class(remaining_vids, novel_classes_ids, K)

print()
# Count the number of instances we go
for cls, cnts in accumulated_counts.items():
    print(f"{cnts} instances for class: {cls}")


# Random order of classes - than start by sampling videos for first class - for the next ones already count the videos decided
# If there is too many for some class, just do not count those ones or something like that 
# You have to exclude them randomly
        
# print(gttubes)  

25 instances for class: 2
15 instances for class: 0
15 instances for class: 4
16 instances for class: 8
15 instances for class: 10

8 instances for class: 2
5 instances for class: 0
5 instances for class: 4
5 instances for class: 8
5 instances for class: 10


# The actual evaluation

In [126]:
from ultralytics import YOLO

detection_model = YOLO("yolo26n.pt")

In [127]:
# Feature extraction model
def get_clip_features(frames, model):
    model.eval()
    with torch.no_grad():

        frame_features = model(frames)

        video_features = torch.mean(frame_features, dim=0)

    return video_features

In [ ]:
# Evaluation loop
video = query_videos[0]

E = 1000

model = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")

for episode in range(E):
    # Sample the query and support videos
    query_videos, _ = get_m_vids_from_novel_class(test_videos, novel_classes_ids, M)
    remaining_vids = list(set(test_videos) - set(query_videos))
    support_videos, _ = get_m_vids_from_novel_class(remaining_vids, novel_classes_ids, K)

    # Build prototypes
    prototypes = {id:[] for id in novel_classes_ids}
    for id in novel_classes_ids:
        for vid in support_videos:
        prototype_features = 
    


basketball/v_5ekaksddqrc_c007
